In [1]:
import pandas as pd

In [2]:
vektis = pd.read_csv("../data/raw/Vektis Open Databestand Zorgverzekeringswet 2024 - gemeente.csv", sep=";")

In [3]:
vektis.head()

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15
1,M,0 t/m 4 jaar,AA EN HUNZE,503,462.82,594197.18,32444.90,35500.43,20536.52,15357.50,...,0.00,0.0,2453.22,0.00,0.00,0.0,23236.19,0.00,0.00,0.00
2,M,0 t/m 4 jaar,AALSMEER,731,665.75,1829500.58,55985.58,51112.38,25609.76,17045.38,...,0.00,0.0,24028.45,0.00,0.00,0.0,125837.16,0.00,0.00,0.00
3,M,0 t/m 4 jaar,AALTEN,662,596.80,862338.30,45558.63,45592.74,27044.28,7194.71,...,0.00,0.0,1549.40,0.00,0.00,0.0,123649.13,0.00,0.00,0.00
4,M,0 t/m 4 jaar,ACHTKARSPELEN,680,619.29,903357.49,41051.42,46985.28,21486.90,11042.53,...,0.00,0.0,1147.02,0.00,0.00,0.0,46237.53,0.00,0.00,0.00


In [ ]:
# Privacy bucket: groups under 10 people (hidden so no one can be identified)
# plus people with unknown sex, age or municipality. All lumped into one unlabelled row.
bucket = vektis[vektis['gemeentenaam'].isna()]
bucket

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15


In [5]:
bucket_people = bucket["aantal_bsn"].sum()
total_people = vektis["aantal_bsn"].sum()
share = bucket_people / total_people * 100
print(f"Bucket: {bucket_people:,.0f} people, {share:.1f}% of all people")

Bucket: 412,394 people, 2.2% of all people


In [45]:
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [46]:
bucket_cost = bucket.filter(like="kosten").sum().sum()
bucket_cost
cost_share = bucket_cost / total_costs * 100
print(f"Bucket: €{bucket_cost:,.0f}, {cost_share:.1f}% of all cost")

Bucket: €167,835,544, 0.3% of all cost


**Decision: privacy bucket**
- Found: one row with empty sex, age and municipality. Vektis puts groups under 10 people (privacy) and people with unknown sex, age or municipality here. It holds 412,394 people (2.2%) but only €167.8 million (0.3% of all cost).
- Did: removed the row. The raw file stays unchanged.
- Why: every KPI needs a municipality, and expected cost needs age and sex. This row has none of them, so it cannot be placed anywhere. Rejected: spreading it over municipalities, because we do not know where these people live.
- Effect: only 5 small cells are missing (Rozendaal, Schiermonnikoog, Vlieland). The bucket is cheap per person (about €760 per insured-year against €3,382), likely because many were insured for only part of the year. Removing it barely changes the national figures.


In [47]:
vektis.head()

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15
1,M,0 t/m 4 jaar,AA EN HUNZE,503,462.82,594197.18,32444.90,35500.43,20536.52,15357.50,...,0.00,0.0,2453.22,0.00,0.00,0.0,23236.19,0.00,0.00,0.00
2,M,0 t/m 4 jaar,AALSMEER,731,665.75,1829500.58,55985.58,51112.38,25609.76,17045.38,...,0.00,0.0,24028.45,0.00,0.00,0.0,125837.16,0.00,0.00,0.00
3,M,0 t/m 4 jaar,AALTEN,662,596.80,862338.30,45558.63,45592.74,27044.28,7194.71,...,0.00,0.0,1549.40,0.00,0.00,0.0,123649.13,0.00,0.00,0.00
4,M,0 t/m 4 jaar,ACHTKARSPELEN,680,619.29,903357.49,41051.42,46985.28,21486.90,11042.53,...,0.00,0.0,1147.02,0.00,0.00,0.0,46237.53,0.00,0.00,0.00


In [49]:
vektis = vektis[vektis["gemeentenaam"].notna()]

In [51]:
vektis.shape

(12991, 31)

In [ ]:
assert len(vektis) == 12991